In [2]:
# =========================================================
# 0️⃣ LIBRERÍAS
# =========================================================
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from statsmodels.tsa.arima.model import ARIMA
from arch import arch_model


# =========================================================
# 1️⃣ DATAFRAME (TU CÓDIGO – SIN CAMBIOS)
# =========================================================
close_prices = np.array([
    [100, 200, 50, 150, 75],
    [102, 198, 52, 152, 77],
    [101, 202, 51, 149, 76],
    [103, 205, 53, 154, 78],
    [104, 207, 54, 156, 80]
])

volumes = np.array([
    [1000, 5000, 3000, 2000, 1500],
    [1200, 5200, 3100, 2200, 1600],
    [1100, 5100, 3050, 2100, 1550],
    [1150, 5300, 3200, 2300, 1650],
    [1180, 5400, 3300, 2400, 1700]
])

tickers = ['Activo A', 'Activo B', 'Activo C', 'Activo D', 'Activo E']

dates = pd.date_range(start='2024-01-01', periods=5, freq='D')

columns = pd.MultiIndex.from_product(
    [['price', 'volume'], tickers],
    names=['type', 'asset']
)

df = pd.DataFrame(
    np.hstack([close_prices, volumes]),
    index=dates,
    columns=columns
)


# =========================================================
# 2️⃣ PREPROCESADO FINANCIERO
# =========================================================
prices_df = df['price']
volumes_df = df['volume']

log_returns = np.log(prices_df / prices_df.shift(1)).dropna()
returns = log_returns.values


# =========================================================
# 3️⃣ RANDOM WALK (BENCHMARK)
# =========================================================
rw_returns = pd.Series(0.0, index=prices_df.columns)


# =========================================================
# 4️⃣ ARIMA SOBRE RETORNOS
# =========================================================
def arima_forecast(series):
    model = ARIMA(series, order=(1, 0, 1))
    fit = model.fit()
    return fit.forecast(1).iloc[0]

arima_returns = log_returns.apply(arima_forecast)


# =========================================================
# 5️⃣ ENSEMBLE CONSERVADOR
# =========================================================
expected_returns = 0.7 * rw_returns + 0.3 * arima_returns
mean_returns = expected_returns.values


# =========================================================
# 6️⃣ GARCH → VOLATILIDAD
# =========================================================
def garch_vol(series):
    model = arch_model(series * 100, vol='Garch', p=1, q=1)
    fit = model.fit(disp='off')
    forecast = fit.forecast(horizon=1)
    return np.sqrt(forecast.variance.values[-1, 0]) / 100

vol_forecast = log_returns.apply(garch_vol)


# =========================================================
# 7️⃣ MATRIZ DE COVARIANZA AJUSTADA
# =========================================================
corr = log_returns.corr()
cov_matrix = np.outer(vol_forecast, vol_forecast) * corr.values


# =========================================================
# 8️⃣ CVaR (RIESGO EXTREMO)
# =========================================================
def cvar(portfolio_returns, alpha=0.05):
    var = np.percentile(portfolio_returns, alpha * 100)
    return -portfolio_returns[portfolio_returns <= var].mean()


# =========================================================
# 9️⃣ OPTIMIZACIÓN DE CARTERA (TU LÓGICA)
# =========================================================
num_assets = prices_df.shape[1]

def portfolio_volatility(weights, cov_matrix):
    return np.sqrt(weights.T @ cov_matrix @ weights)

def portfolio_return(weights, mean_returns):
    return weights @ mean_returns

def complex_risk(weights, cov_matrix, returns, volumes,
                 alpha=0.5, beta=0.3, gamma=0.2):

    vol = portfolio_volatility(weights, cov_matrix)
    vol_penalty = np.sum(weights / (volumes.mean(axis=0) + 1e-6))
    port_returns = returns @ weights
    risk_cvar = cvar(port_returns)

    return alpha * vol + beta * vol_penalty + gamma * risk_cvar

def negative_portfolio_return(weights, mean_returns):
    return -portfolio_return(weights, mean_returns)

target_risk = 0.05

constraints = [
    {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
    {'type': 'ineq',
     'fun': lambda w: target_risk -
     complex_risk(w, cov_matrix, returns, volumes_df.values)}
]

bounds = [(0, 0.9)] * num_assets
init_guess = np.ones(num_assets) / num_assets

result = minimize(
    negative_portfolio_return,
    init_guess,
    args=(mean_returns,),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

weights = result.x
weights[weights < 0.03] = 0
weights /= weights.sum()


# =========================================================
# 🔟 RESULTADOS
# =========================================================
print("\n📊 CARTERA FINAL OPTIMIZADA:\n")

for asset, w in zip(tickers, weights):
    print(f"{asset}: {w:.2%}")

final_return = portfolio_return(weights, mean_returns)
final_risk = complex_risk(weights, cov_matrix, returns, volumes_df.values)

print(f"\nRentabilidad esperada diaria: {final_return:.4%}")
print(f"Riesgo total (vol + liquidez + CVaR): {final_risk:.4f}")


c:\Users\jorge\miniconda3\envs\proyecto_env\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\jorge\miniconda3\envs\proyecto_env\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\jorge\miniconda3\envs\proyecto_env\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "



📊 CARTERA FINAL OPTIMIZADA:

Activo A: 0.00%
Activo B: 63.46%
Activo C: 36.54%
Activo D: 0.00%
Activo E: 0.00%

Rentabilidad esperada diaria: 0.2799%
Riesgo total (vol + liquidez + CVaR): 0.0019


c:\Users\jorge\miniconda3\envs\proyecto_env\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
